In [ ]:
# xT model for passing and carries, JSON-only
# Builds a Karun Singh-style expected threat grid from Opta events,
# then values successful passes and inferred carries/take-ons.

from pathlib import Path
import json
from datetime import datetime
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

SOURCE_DIR = Path('/Users/marclamberts/Event data/Eredivisie')
OUTPUT_DIR = SOURCE_DIR / 'xT'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GRID_X_BINS = 16
GRID_Y_BINS = 12
VALUE_ITERATIONS = 250
CONVERGENCE_TOL = 1e-8
MIN_MOVE_DISTANCE = 3.0
MAX_CARRY_DISTANCE = 60.0
MAX_CARRY_SECONDS = 12
CREDIT = 'Source: Opta/StatsPerform | Models by Marc Lamberts - Waltzing Analytics | July 9, 2026'

SHOT_TYPE_IDS = {13, 14, 15, 16}
PASS_TYPE_IDS = {1, 2}
TAKEON_TYPE_IDS = {3}
TOUCH_EVENT_IDS = {1, 2, 3, 4, 5, 6, 7, 8, 12, 13, 14, 15, 16, 43, 44, 45, 49, 50, 52, 55, 61, 74, 83}
Q_END_X = 140
Q_END_Y = 141
Q_PASS_LENGTH = 212
Q_PASS_ANGLE = 213


def safe_int(value, default=0):
    try:
        return int(value)
    except Exception:
        return default


def safe_float(value, default=np.nan):
    try:
        return float(value)
    except Exception:
        return default


def get_q(event, qid, default=None):
    for q in event.get('qualifier', []) or []:
        if safe_int(q.get('qualifierId'), -1) == qid:
            return q.get('value', default)
    return default


def event_sort_key(event, fallback_index=0):
    return (
        safe_int(event.get('periodId')),
        safe_int(event.get('timeMin')),
        safe_int(event.get('timeSec')),
        safe_int(event.get('eventId'), fallback_index),
        fallback_index,
    )


def event_seconds(event):
    return safe_int(event.get('timeMin')) * 60 + safe_int(event.get('timeSec'))


def zone_id(x, y):
    x = float(np.clip(safe_float(x, 50), 0, 100 - 1e-6))
    y = float(np.clip(safe_float(y, 50), 0, 100 - 1e-6))
    xb = int(np.floor(x / 100 * GRID_X_BINS))
    yb = int(np.floor(y / 100 * GRID_Y_BINS))
    return yb * GRID_X_BINS + xb


def zone_center(z):
    xb = z % GRID_X_BINS
    yb = z // GRID_X_BINS
    return ((xb + 0.5) * 100 / GRID_X_BINS, (yb + 0.5) * 100 / GRID_Y_BINS)


def distance(a, b, c, d):
    return float(np.hypot(safe_float(c) - safe_float(a), safe_float(d) - safe_float(b)))


def parse_match_names(path):
    stem = path.stem
    if len(stem) > 11 and ' - ' in stem[11:]:
        home, away = stem[11:].split(' - ', 1)
    else:
        home, away = '', ''
    return stem, home, away


def load_events(path):
    with path.open('r', encoding='utf-8-sig') as f:
        payload = json.load(f)
    events = payload.get('liveData', {}).get('event', payload.get('event', [])) if isinstance(payload, dict) else []
    return sorted(events if isinstance(events, list) else [], key=lambda e: event_sort_key(e))


def action_type_name(type_id):
    if type_id in PASS_TYPE_IDS:
        return 'pass'
    if type_id in TAKEON_TYPE_IDS:
        return 'take_on'
    if type_id in SHOT_TYPE_IDS:
        return 'shot'
    return 'other'


json_files = sorted(p for p in SOURCE_DIR.glob('*.json') if p.is_file())
if not json_files:
    raise FileNotFoundError(f'No JSON files found in {SOURCE_DIR}')

rows = []
for path in json_files:
    events = load_events(path)
    match_name, home_name, away_name = parse_match_names(path)
    for idx, e in enumerate(events):
        type_id = safe_int(e.get('typeId'), -1)
        team_id = str(e.get('contestantId', ''))
        player_id = str(e.get('playerId', ''))
        x = safe_float(e.get('x'))
        y = safe_float(e.get('y'))
        if not team_id or pd.isna(x) or pd.isna(y):
            continue
        end_x = safe_float(get_q(e, Q_END_X))
        end_y = safe_float(get_q(e, Q_END_Y))
        if pd.isna(end_x) or pd.isna(end_y):
            end_x, end_y = x, y
        rows.append({
            'match_file': path.name,
            'match_name': match_name,
            'event_id': str(e.get('id', e.get('eventId', f'{path.name}-{idx}'))),
            'event_index': safe_int(e.get('eventId'), idx),
            'period_id': safe_int(e.get('periodId')),
            'time_min': safe_int(e.get('timeMin')),
            'time_sec': safe_int(e.get('timeSec')),
            'time_seconds': event_seconds(e),
            'contestant_id': team_id,
            'player_id': player_id,
            'player_name': e.get('playerName', ''),
            'type_id': type_id,
            'event_type': action_type_name(type_id),
            'outcome': safe_int(e.get('outcome'), 0),
            'x': x,
            'y': y,
            'end_x': end_x,
            'end_y': end_y,
            'start_zone': zone_id(x, y),
            'end_zone': zone_id(end_x, end_y),
            'is_shot': int(type_id in SHOT_TYPE_IDS),
            'is_goal': int(type_id == 16),
            'pass_length': safe_float(get_q(e, Q_PASS_LENGTH)),
            'pass_angle': safe_float(get_q(e, Q_PASS_ANGLE)),
        })

events_df = pd.DataFrame(rows).sort_values(['match_file', 'period_id', 'time_min', 'time_sec', 'event_index']).reset_index(drop=True)
if events_df.empty:
    raise RuntimeError('No coordinate events loaded.')
print(f'Loaded {len(events_df):,} coordinate events from {len(json_files):,} matches')

# Team names: infer a stable contestant_id -> club name by counting names appearing
# in all fixture filenames for that contestant. The actual club recurs every match;
# opponents only recur occasionally.
team_name_votes = []
for match_file, g in events_df.groupby('match_file', sort=False):
    stem = Path(match_file).stem
    names = stem[11:].split(' - ', 1) if len(stem) > 11 and ' - ' in stem[11:] else []
    teams = g['contestant_id'].drop_duplicates().tolist()
    for tid in teams:
        for name in names:
            team_name_votes.append({'contestant_id': tid, 'candidate_team_name': name})
if team_name_votes:
    votes = pd.DataFrame(team_name_votes)
    team_map = (
        votes.groupby(['contestant_id', 'candidate_team_name']).size().rename('votes').reset_index()
        .sort_values(['contestant_id', 'votes'], ascending=[True, False])
        .drop_duplicates('contestant_id')
        .rename(columns={'candidate_team_name': 'team_name'})[['contestant_id', 'team_name']]
    )
else:
    team_map = pd.DataFrame(columns=['contestant_id', 'team_name'])
events_df = events_df.merge(team_map, on='contestant_id', how='left')
events_df['team_name'] = events_df['team_name'].fillna(events_df['contestant_id'])

# -----------------------------------------------------------------------------
# Build xT state model from shots and successful move transitions.
# -----------------------------------------------------------------------------
shots = events_df[events_df['is_shot'].eq(1)].copy()
shot_counts = shots.groupby('start_zone').size().reindex(range(GRID_X_BINS * GRID_Y_BINS), fill_value=0)
goal_counts = shots.groupby('start_zone')['is_goal'].sum().reindex(range(GRID_X_BINS * GRID_Y_BINS), fill_value=0)
all_event_counts = events_df.groupby('start_zone').size().reindex(range(GRID_X_BINS * GRID_Y_BINS), fill_value=0)

# Successful passes and take-ons are observed moves for the transition matrix.
passes = events_df[events_df['type_id'].isin(PASS_TYPE_IDS) & events_df['outcome'].eq(1)].copy()
passes['move_type'] = 'pass'
passes['move_distance'] = np.hypot(passes['end_x'] - passes['x'], passes['end_y'] - passes['y'])
passes = passes[passes['move_distance'] >= MIN_MOVE_DISTANCE]

takeons = events_df[events_df['type_id'].isin(TAKEON_TYPE_IDS) & events_df['outcome'].eq(1)].copy()
takeons['move_type'] = 'take_on'
takeons['move_distance'] = np.hypot(takeons['end_x'] - takeons['x'], takeons['end_y'] - takeons['y'])
takeons = takeons[takeons['move_distance'] >= MIN_MOVE_DISTANCE]

# Inferred carries: same player/team keeps the ball between consecutive touches and changes location.
touch = events_df[events_df['type_id'].isin(TOUCH_EVENT_IDS) & events_df['player_id'].astype(str).ne('')].copy()
touch['next_match_file'] = touch['match_file'].shift(-1)
touch['next_period_id'] = touch['period_id'].shift(-1)
touch['next_team'] = touch['contestant_id'].shift(-1)
touch['next_player_id'] = touch['player_id'].shift(-1)
touch['next_player_name'] = touch['player_name'].shift(-1)
touch['next_time_seconds'] = touch['time_seconds'].shift(-1)
touch['next_x'] = touch['x'].shift(-1)
touch['next_y'] = touch['y'].shift(-1)
touch['next_zone'] = touch['start_zone'].shift(-1)
carry_mask = (
    touch['match_file'].eq(touch['next_match_file'])
    & touch['period_id'].eq(touch['next_period_id'])
    & touch['contestant_id'].eq(touch['next_team'])
    & touch['player_id'].eq(touch['next_player_id'])
    & touch['next_time_seconds'].sub(touch['time_seconds']).between(1, MAX_CARRY_SECONDS)
)
carry = touch[carry_mask].copy()
carry['move_distance'] = np.hypot(carry['next_x'] - carry['end_x'], carry['next_y'] - carry['end_y'])
carry = carry[carry['move_distance'].between(MIN_MOVE_DISTANCE, MAX_CARRY_DISTANCE)].copy()
carry['move_type'] = 'carry'
carry['x'] = carry['end_x']
carry['y'] = carry['end_y']
carry['end_x'] = carry['next_x']
carry['end_y'] = carry['next_y']
carry['start_zone'] = carry.apply(lambda r: zone_id(r['x'], r['y']), axis=1)
carry['end_zone'] = carry['next_zone'].astype(int)
carry['event_id'] = carry['event_id'].astype(str) + '_carry'
carry['event_type'] = 'carry'

moves = pd.concat([
    passes[events_df.columns.tolist() + ['move_type', 'move_distance']],
    takeons[events_df.columns.tolist() + ['move_type', 'move_distance']],
    carry[events_df.columns.tolist() + ['move_type', 'move_distance']],
], ignore_index=True, sort=False)

n_states = GRID_X_BINS * GRID_Y_BINS
move_counts = moves.groupby('start_zone').size().reindex(range(n_states), fill_value=0)
p_shot = (shot_counts / (shot_counts + move_counts).replace(0, np.nan)).fillna(0).clip(0, 1).to_numpy(float)
p_move = 1 - p_shot
league_goal_rate = shots['is_goal'].mean() if len(shots) else 0.10
shot_goal_prob = ((goal_counts + league_goal_rate * 20) / (shot_counts + 20)).fillna(league_goal_rate).to_numpy(float)

T = np.zeros((n_states, n_states), dtype=float)
for (s, e), count in moves.groupby(['start_zone', 'end_zone']).size().items():
    T[int(s), int(e)] += count
row_sums = T.sum(axis=1, keepdims=True)
T = np.divide(T, row_sums, out=np.zeros_like(T), where=row_sums > 0)

xT = np.zeros(n_states, dtype=float)
for _ in range(VALUE_ITERATIONS):
    new_xT = p_shot * shot_goal_prob + p_move * T.dot(xT)
    if np.max(np.abs(new_xT - xT)) < CONVERGENCE_TOL:
        xT = new_xT
        break
    xT = new_xT

zone_values = pd.DataFrame({
    'zone': np.arange(n_states),
    'zone_x': [zone_center(z)[0] for z in range(n_states)],
    'zone_y': [zone_center(z)[1] for z in range(n_states)],
    'event_count': all_event_counts.values,
    'shot_count': shot_counts.values,
    'move_count': move_counts.values,
    'p_shot': p_shot,
    'shot_goal_prob': shot_goal_prob,
    'xT': xT,
})

moves['start_xT'] = xT[moves['start_zone'].astype(int).clip(0, n_states - 1).to_numpy()]
moves['end_xT'] = xT[moves['end_zone'].astype(int).clip(0, n_states - 1).to_numpy()]
moves['xT_added'] = moves['end_xT'] - moves['start_xT']
moves['positive_xT_added'] = moves['xT_added'].clip(lower=0)

# -----------------------------------------------------------------------------
# Summaries
# -----------------------------------------------------------------------------
player_summary = (
    moves.groupby(['player_id', 'player_name', 'contestant_id', 'team_name'], dropna=False)
    .agg(
        actions=('event_id', 'count'),
        passes=('move_type', lambda s: int((s == 'pass').sum())),
        carries=('move_type', lambda s: int((s == 'carry').sum())),
        take_ons=('move_type', lambda s: int((s == 'take_on').sum())),
        xT_added=('xT_added', 'sum'),
        positive_xT_added=('positive_xT_added', 'sum'),
        xT_per_action=('xT_added', 'mean'),
        avg_move_distance=('move_distance', 'mean'),
    )
    .reset_index()
)
player_summary['xT_per_100_actions'] = player_summary['xT_added'] / player_summary['actions'].replace(0, np.nan) * 100
player_summary['positive_xT_per_100_actions'] = player_summary['positive_xT_added'] / player_summary['actions'].replace(0, np.nan) * 100
player_summary = player_summary.sort_values('xT_added', ascending=False)

team_summary = (
    moves.groupby(['contestant_id', 'team_name'], dropna=False)
    .agg(
        actions=('event_id', 'count'),
        passes=('move_type', lambda s: int((s == 'pass').sum())),
        carries=('move_type', lambda s: int((s == 'carry').sum())),
        take_ons=('move_type', lambda s: int((s == 'take_on').sum())),
        xT_added=('xT_added', 'sum'),
        positive_xT_added=('positive_xT_added', 'sum'),
        xT_per_action=('xT_added', 'mean'),
    )
    .reset_index()
    .sort_values('xT_added', ascending=False)
)

move_type_summary = (
    moves.groupby('move_type')
    .agg(actions=('event_id', 'count'), xT_added=('xT_added', 'sum'), positive_xT_added=('positive_xT_added', 'sum'), xT_per_action=('xT_added', 'mean'))
    .reset_index()
    .sort_values('xT_added', ascending=False)
)

# Save CSVs.
action_cols = [
    'match_file', 'match_name', 'event_id', 'event_index', 'period_id', 'time_min', 'time_sec', 'time_seconds',
    'contestant_id', 'team_name', 'player_id', 'player_name', 'move_type', 'type_id', 'outcome',
    'x', 'y', 'end_x', 'end_y', 'start_zone', 'end_zone', 'move_distance', 'start_xT', 'end_xT', 'xT_added', 'positive_xT_added'
]
moves[action_cols].to_csv(OUTPUT_DIR / 'xt_action_values.csv', index=False)
player_summary.to_csv(OUTPUT_DIR / 'xt_player_summary.csv', index=False)
team_summary.to_csv(OUTPUT_DIR / 'xt_team_summary.csv', index=False)
zone_values.to_csv(OUTPUT_DIR / 'xt_grid_values.csv', index=False)
move_type_summary.to_csv(OUTPUT_DIR / 'xt_move_type_summary.csv', index=False)

meta = {
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'source': str(SOURCE_DIR),
    'grid_x_bins': GRID_X_BINS,
    'grid_y_bins': GRID_Y_BINS,
    'method': 'Karun Singh-style xT: xT = P(shot)*P(goal|shot) + P(move)*sum(P(destination)*xT(destination))',
    'move_types': {
        'pass': 'successful Opta pass events with end coordinates',
        'carry': 'inferred same-player same-team ball movement between consecutive touches',
        'take_on': 'successful Opta take-on movement where end coordinates imply movement',
    },
    'outputs': ['xt_action_values.csv', 'xt_player_summary.csv', 'xt_team_summary.csv', 'xt_grid_values.csv', 'xt_move_type_summary.csv'],
}
with (OUTPUT_DIR / 'xt_model_meta.json').open('w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)

# -----------------------------------------------------------------------------
# Core visuals
# -----------------------------------------------------------------------------
def style_ax(ax):
    ax.set_facecolor('#0e1117')
    for sp in ax.spines.values():
        sp.set_color('#30363d')
    ax.tick_params(colors='#8b949e', labelsize=9)
    ax.grid(color='#30363d', alpha=.45, lw=.6)


def footer(fig, note=''):
    if note:
        fig.text(.045, .030, note, color='#6e7681', fontsize=8, ha='left', va='bottom')
    fig.text(.955, .012, CREDIT, color='#8b949e', fontsize=8.5, ha='right', va='bottom')


def save_bar(data, value, label, title, filename, top=25, subtitle=''):
    d = data.dropna(subset=[value]).sort_values(value, ascending=False).head(top).sort_values(value)
    fig, ax = plt.subplots(figsize=(13, 9), dpi=180)
    fig.patch.set_facecolor('#0e1117')
    style_ax(ax)
    ax.barh(d[label], d[value], color='#58a6ff')
    ax.set_xlabel(value.replace('_', ' ').title(), color='#e6edf3', fontweight='bold')
    fig.text(.045, .965, title, color='white', fontsize=18, fontweight='bold', ha='left', va='top')
    if subtitle:
        fig.text(.045, .928, subtitle, color='#8b949e', fontsize=10, ha='left', va='top')
    footer(fig, 'xT values are for successful passes, inferred carries and successful take-ons.')
    fig.savefig(OUTPUT_DIR / filename, facecolor=fig.get_facecolor(), bbox_inches='tight', pad_inches=.18)
    plt.close(fig)

player_summary['short'] = player_summary['player_name'].fillna('').str.slice(0, 24)
save_bar(player_summary, 'xT_added', 'short', 'Top Players by Total xT Added', 'xt_top_players_total.png')
save_bar(player_summary[player_summary['actions'] >= 200], 'xT_per_100_actions', 'short', 'Top Players by xT per 100 Actions', 'xt_top_players_per100.png', subtitle='Minimum 200 pass/carry actions')
save_bar(team_summary, 'xT_added', 'team_name', 'Team xT Added from Passes and Carries', 'xt_team_totals.png', top=18)

# xT grid heatmap.
fig, ax = plt.subplots(figsize=(12, 8), dpi=180)
fig.patch.set_facecolor('#0e1117')
style_ax(ax)
grid = np.full((GRID_Y_BINS, GRID_X_BINS), np.nan)
for _, r in zone_values.iterrows():
    grid[int(r['zone']) // GRID_X_BINS, int(r['zone']) % GRID_X_BINS] = r['xT']
im = ax.imshow(grid, origin='lower', extent=[0,100,0,100], cmap='viridis', aspect='auto')
ax.set_xlabel('Pitch X', color='#e6edf3', fontweight='bold')
ax.set_ylabel('Pitch Y', color='#e6edf3', fontweight='bold')
fig.text(.045, .965, 'xT Pitch Zone Values', color='white', fontsize=18, fontweight='bold', ha='left', va='top')
fig.text(.045, .928, f'{GRID_X_BINS} x {GRID_Y_BINS} grid learned from JSON events', color='#8b949e', fontsize=10, ha='left', va='top')
cb = fig.colorbar(im, ax=ax, fraction=.035, pad=.015)
cb.ax.tick_params(colors='#8b949e')
cb.set_label('xT', color='#e6edf3')
cb.outline.set_edgecolor('#30363d')
footer(fig, 'Higher zones indicate greater expected scoring threat from possession.')
fig.savefig(OUTPUT_DIR / 'xt_grid_heatmap.png', facecolor=fig.get_facecolor(), bbox_inches='tight', pad_inches=.18)
plt.close(fig)

# Move type comparison.
fig, ax = plt.subplots(figsize=(10, 6), dpi=180)
fig.patch.set_facecolor('#0e1117')
style_ax(ax)
mt = move_type_summary.sort_values('xT_added')
ax.barh(mt['move_type'], mt['xT_added'], color=['#58a6ff', '#3fb950', '#f78166'][:len(mt)])
ax.set_xlabel('Total xT Added', color='#e6edf3', fontweight='bold')
fig.text(.045, .965, 'xT by Move Type', color='white', fontsize=18, fontweight='bold', ha='left', va='top')
footer(fig, 'Passes are explicit Opta events; carries are inferred between same-player touches.')
fig.savefig(OUTPUT_DIR / 'xt_by_move_type.png', facecolor=fig.get_facecolor(), bbox_inches='tight', pad_inches=.18)
plt.close(fig)

print('\nSaved xT model outputs to:', OUTPUT_DIR)
for name in meta['outputs'] + ['xt_model_meta.json', 'xt_top_players_total.png', 'xt_top_players_per100.png', 'xt_team_totals.png', 'xt_grid_heatmap.png', 'xt_by_move_type.png']:
    p = OUTPUT_DIR / name
    print(f'  {name:28s} {p.stat().st_size / 1024:9.1f} KB')
print('\nMove type summary:')
print(move_type_summary.round(5).to_string(index=False))
print('\nTop players by xT added:')
print(player_summary[['player_name', 'team_name', 'actions', 'passes', 'carries', 'take_ons', 'xT_added', 'xT_per_100_actions']].head(20).round(4).to_string(index=False))
